<a href="https://colab.research.google.com/github/vignesh-potharaj/gen-ai/blob/main/HealthCareData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install Google GenAI SDK, FAISS for vector search, sentence-transformers for embeddings, and text splitters
!pip install -q -U google-genai faiss-cpu sentence-transformers langchain-text-splitters

import os
import faiss
import numpy as np
from google.colab import userdata
from google.genai import types, client
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Retrieve Gemini API Key from Google Colab Secrets
api_key = userdata.get('GEMINI_API_KEY')

# Initialize Gemini Client
ai = client.Client(api_key=api_key)

# Initialize open-source embedding model for FAISS indexing
embedder = SentenceTransformer('all-MiniLM-L6-v2')

print("Environment setup complete! Gemini client and Embedding model initialized.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 809.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.7/596.7 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 258.6/258.6 kB 14.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.56.2 which is incompatible.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Environment setup complete! Gemini client and Embedding model initialized.


In [2]:
# 1. Define synthetic long patient records
patient_records = [
    {
        "patient_id": "P-101",
        "patient_name": "Eleanor Vance",
        "date": "2026-03-12",
        "diagnosis": "Type 2 Diabetes & Hypertension",
        "text": """Patient Eleanor Vance presented on March 12, 2026, for a quarterly chronic disease evaluation.
        Subjective: Patient reports mild fatigue and slight dizziness in the mornings. Blood glucose logs show fasting levels averaging 145 mg/dL.
        Objective: BP 138/88 mmHg, HR 72 bpm, HbA1c measured at 7.8%. Kidney function tests (eGFR) remain normal at 85 mL/min.
        Assessment: Suboptimally controlled Type 2 Diabetes Mellitus with mild stage 1 hypertension.
        Plan: Increased Metformin dosage from 500mg BID to 1000mg BID. Continued Lisinopril 10mg daily for blood pressure control.
        Advised lifestyle modification including 30 minutes of moderate daily exercise and strict adherence to a low-glycemic diet. Follow-up in 8 weeks."""
    },
    {
        "patient_id": "P-102",
        "patient_name": "Arthur Pendelton",
        "date": "2026-04-05",
        "diagnosis": "Acute Asthma Exacerbation",
        "text": """Patient Arthur Pendelton presented on April 5, 2026, with acute shortness of breath, bilateral wheezing, and persistent dry cough.
        Subjective: Symptoms began two days ago following seasonal allergy flare-ups. Albuterol rescue inhaler provided only temporary relief.
        Objective: Oxygen saturation 93% on room air, RR 24/min, HR 98 bpm. Chest auscultation revealed moderate expiratory wheezing throughout all lung fields.
        Assessment: Moderate acute asthma exacerbation triggered by seasonal allergens.
        Plan: Administered nebulized Albuterol/Ipratropium in clinic with immediate improvement in SpO2 to 97%.
        Prescribed a 5-day oral Prednisone taper (40mg daily) and daily Fluticasone propionate inhaler. Advised to avoid pollen exposure and return if distress recurs."""
    }
]

# 2. Initialize recursive text splitter for chunking
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " "]
)

# 3. Chunk documents and attach metadata
processed_chunks = []

for record in patient_records:
    chunks = text_splitter.split_text(record["text"])
    for i, chunk_text in enumerate(chunks):
        chunk_entry = {
            "chunk_id": f"{record['patient_id']}_c{i+1}",
            "patient_id": record["patient_id"],
            "patient_name": record["patient_name"],
            "date": record["date"],
            "diagnosis": record["diagnosis"],
            "content": chunk_text.strip()
        }
        processed_chunks.append(chunk_entry)

# Print summary of chunking results
print(f"Total Patient Records Processed: {len(patient_records)}")
print(f"Total Chunks Created: {len(processed_chunks)}\n")

# Display a sample chunk with attached metadata
print("=== SAMPLE CHUNK WITH METADATA ===")
print(f"Chunk ID:    {processed_chunks[0]['chunk_id']}")
print(f"Patient ID:  {processed_chunks[0]['patient_id']} ({processed_chunks[0]['patient_name']})")
print(f"Date:        {processed_chunks[0]['date']}")
print(f"Diagnosis:   {processed_chunks[0]['diagnosis']}")
print(f"Content:     {processed_chunks[0]['content']}")

Total Patient Records Processed: 2
Total Chunks Created: 9

=== SAMPLE CHUNK WITH METADATA ===
Chunk ID:    P-101_c1
Patient ID:  P-101 (Eleanor Vance)
Date:        2026-03-12
Diagnosis:   Type 2 Diabetes & Hypertension
Content:     Patient Eleanor Vance presented on March 12, 2026, for a quarterly chronic disease evaluation.
        Subjective: Patient reports mild fatigue and slight dizziness in the mornings. Blood glucose logs show fasting levels averaging 145 mg/dL.


In [3]:
# Extract all chunk text content
chunk_texts = [chunk["content"] for chunk in processed_chunks]

# 1. Generate dense vector embeddings for each chunk
embeddings = embedder.encode(chunk_texts, convert_to_numpy=True)

# 2. Normalize embeddings for cosine similarity searching
faiss.normalize_L2(embeddings)

# 3. Initialize FAISS index based on vector dimension (384 for MiniLM)
embedding_dim = embeddings.shape[1]
index = faiss.IndexFlatIP(embedding_dim)  # Inner Product (IP) on normalized vectors = Cosine Similarity

# 4. Add embeddings to FAISS index
index.add(embeddings)

print(f"Successfully generated embeddings for {len(chunk_texts)} chunks.")
print(f"FAISS Index built! Total vectors indexed: {index.ntotal}")

Successfully generated embeddings for 9 chunks.
FAISS Index built! Total vectors indexed: 9
